# 스태킹 앙상블 파이프라인
OOF 생성 → Meta LR → OOF 스코어 저장 → Test 평가 → 요약

- **FORCE_RERUN = False**: 캐시 있으면 스킵
- **FORCE_RERUN = True**: 항상 재실행

In [1]:
# Cell 1: 라이브러리
import json, joblib, math, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
print('라이브러리 로드 완료')

라이브러리 로드 완료


In [2]:
# Cell 2: 설정 + 파라미터 + 데이터 로드
RANDOM_STATE = 42
N_SPLITS     = 5
META_GAP_MAX = 0.05
FORCE_RERUN  = True  # True: 캐시 무시하고 전체 재실행

CACHE_DIR = Path('cache')
DATA_DIR  = Path('../_data')
TRAIN_PATH = DATA_DIR / '02_interim' / '260513 feature' / 'Membership_final.csv'

# Test: FastAPI step00이 생성한 test_expanded.csv 우선, 없으면 Membership_final_2.csv
_test_expanded = CACHE_DIR / 'test_expanded.csv'
_test_fallback = DATA_DIR / '02_interim' / '260513 feature' / 'Membership_final_2.csv'
TEST_PATH = _test_expanded if _test_expanded.exists() else _test_fallback
print(f'Test 데이터: {TEST_PATH}')

# ── 파라미터 로드 (step05_base_params.json) ────────────────────────────────────
with open(CACHE_DIR / 'step05_base_params.json', encoding='utf-8') as f:
    _raw = json.load(f)
BASE_PARAMS = {name: info['best_params'] for name, info in _raw.items()}

print('\n파라미터 (step05_base_params.json):')
for name in BASE_PARAMS:
    print(f'  {name:<12} best_auc={_raw[name]["best_auc"]}')

# ── 데이터 로드 ────────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f'\ntrain: {train_df.shape}  /  test: {test_df.shape}')

Test 데이터: cache\test_expanded.csv

파라미터 (step05_base_params.json):
  LR           best_auc=0.8509
  XGBoost      best_auc=0.8854
  CatBoost     best_auc=0.8821
  RF           best_auc=0.8681
  SVM          best_auc=0.8594

train: (23081, 91)  /  test: (2545, 82)


In [3]:
# Cell 3: 공통 함수 정의
import threading

EXCLUDE_BASE = {
    'USER_KEY', 'is_repurchase',
    'reg_date', 'end_date', 'product_code', 'billing_method',
    'payment_device', 'gender', 'age', 'reg_hour', 'price', 'max_screen',
}
MODEL_NAMES  = ['LR', 'XGBoost', 'CatBoost', 'RF', 'SVM']

SCOPES = {
    'overall':           (lambda df: df.copy(),                              True),
    'promotion_only':    (lambda df: df[df['is_promotion'] == 1].copy(),     False),
    'nonpromotion_only': (lambda df: df[df['is_promotion'] == 0].copy(),     False),
}

_print_lock = threading.Lock()
def safe_print(*args, **kwargs):
    with _print_lock:
        print(*args, **kwargs)


def get_features(df, inc_promotion):
    exclude = EXCLUDE_BASE | (set() if inc_promotion else {'is_promotion'})
    return [c for c in df.columns if c not in exclude]


def build_model(name, params, n_jobs=-1):
    p = dict(params)
    if name == 'LR':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(**p, max_iter=3000, random_state=RANDOM_STATE))
        ])
    if name == 'XGBoost':
        return XGBClassifier(
            **p,
            objective='binary:logistic', eval_metric='logloss',
            random_state=RANDOM_STATE, n_jobs=n_jobs
        )
    if name == 'CatBoost':
        thread_count = p.pop('thread_count', -1)
        if n_jobs == 1:
            thread_count = 1
        return CatBoostClassifier(**p, thread_count=thread_count)
    if name == 'RF':
        return RandomForestClassifier(**p, random_state=RANDOM_STATE, n_jobs=n_jobs)
    if name == 'SVM':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(**p, probability=True))
        ])
    raise ValueError(f'Unknown model: {name}')


def run_oof_single(name, params, X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(X), np.nan)
    tr_aucs, va_aucs = [], []
    model_start = time.time()
    for fold_i, (tr, va) in enumerate(skf.split(X, y), 1):
        model = build_model(name, params, n_jobs=1)
        model.fit(X.iloc[tr], y[tr])
        p_va = model.predict_proba(X.iloc[va])[:, 1]
        oof[va] = p_va
        va_auc = roc_auc_score(y[va], p_va) if len(np.unique(y[va])) == 2 else None
        if va_auc is not None:
            va_aucs.append(va_auc)
            safe_print(f'    {name:<12} fold {fold_i}/{N_SPLITS}  val={va_auc:.4f}  ({round(time.time()-model_start)}s 경과)')
        if len(np.unique(y[tr])) == 2:
            tr_aucs.append(roc_auc_score(y[tr], model.predict_proba(X.iloc[tr])[:, 1]))
    valid   = ~np.isnan(oof)
    oof_auc = round(float(roc_auc_score(y[valid], oof[valid])), 4) if len(np.unique(y[valid])) == 2 else None
    gap     = round(float(np.mean(tr_aucs) - np.mean(va_aucs)), 4) if tr_aucs and va_aucs else None
    return {
        'oof': oof,
        'oof_auc': oof_auc,
        'gap': gap,
        'mean_train_auc': round(float(np.mean(tr_aucs)), 4) if tr_aucs else None,
    }


def json_safe(obj):
    if isinstance(obj, float) and (math.isinf(obj) or math.isnan(obj)):
        return None
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    return obj


def save_json(name, data):
    p = CACHE_DIR / f'{name}.json'
    p.write_text(json.dumps(json_safe(data), default=str, indent=2, ensure_ascii=False), encoding='utf-8')


def load_json(name):
    p = CACHE_DIR / f'{name}.json'
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else None


def mark_done(step, meta=None):
    sf = CACHE_DIR / 'pipeline_state.json'
    state = json.loads(sf.read_text(encoding='utf-8')) if sf.exists() else {}
    state[step] = {'completed_at': datetime.now().isoformat(), 'meta': meta or {}}
    sf.write_text(json.dumps(state, default=str, indent=2, ensure_ascii=False), encoding='utf-8')


print('함수 정의 완료')

함수 정의 완료


In [4]:
# Cell 4: OOF 실행 + pkl 저장 + 캐시 저장
_oof_cached = (
    not FORCE_RERUN
    and (CACHE_DIR / 'step05_oof.json').exists()
    and (CACHE_DIR / 'step05_oof_arrays.json').exists()
)

if _oof_cached:
    print('캐시에서 OOF 로드 (재실행하려면 FORCE_RERUN=True)')
    oof_arrays  = load_json('step05_oof_arrays')
    oof_summary = load_json('step05_oof')
else:
    oof_arrays  = {}
    oof_summary = {'by_scope': {}}
    total_start = time.time()

    for scope_name, (scope_fn, inc_promo) in SCOPES.items():
        df_scope = scope_fn(train_df).reset_index(drop=True)
        features = get_features(df_scope, inc_promo)
        X = df_scope[features].apply(pd.to_numeric, errors='coerce').fillna(0)
        y = (1 - df_scope['is_repurchase'].astype(int)).to_numpy()

        print(f'\n{"="*55}')
        print(f'[{scope_name}]  {len(X):,}행  {len(features)}피처  OOF 시작...')
        print(f'{"="*55}')
        scope_start = time.time()

        oof_arrays[scope_name]              = {}
        oof_summary['by_scope'][scope_name] = {}

        # ── 5개 모델 병렬 OOF (폴드별 실시간 출력) ────────────────────────────
        raw_results = {}
        with ThreadPoolExecutor(max_workers=5) as pool:
            futures = {
                pool.submit(run_oof_single, name, BASE_PARAMS[name], X, y): name
                for name in MODEL_NAMES
            }
            for fut in as_completed(futures):
                name = futures[fut]
                try:
                    res = fut.result()
                    raw_results[name] = res
                    overfit_str = '⚠️ True' if (res['gap'] or 0) > META_GAP_MAX else 'False'
                    print(f'  ✅ {name:<12} OOF AUC={res["oof_auc"]}  gap={res["gap"]}  과적합={overfit_str}')
                except Exception as e:
                    raw_results[name] = None
                    print(f'  ❌ {name} 오류: {e}')

        # ── 결과 정리 ──────────────────────────────────────────────────────────
        for name in MODEL_NAMES:
            res = raw_results.get(name)
            if res is None:
                continue
            oof_arrays[scope_name][name] = res['oof'].tolist()
            oof_summary['by_scope'][scope_name][name] = {
                'oof_auc':        res['oof_auc'],
                'gap':            res['gap'],
                'mean_train_auc': res['mean_train_auc'],
                'overfit':        bool((res['gap'] or 0) > META_GAP_MAX),
            }

        # ── 전체 학습 + pkl 저장 ───────────────────────────────────────────────
        print(f'  pkl 저장 중...', end=' ')
        for name in MODEL_NAMES:
            try:
                m = build_model(name, BASE_PARAMS[name], n_jobs=-1)
                m.fit(X, y)
                joblib.dump(m, CACHE_DIR / f'stacking_{name}_{scope_name}.pkl')
                if name == 'CatBoost':
                    joblib.dump(m, CACHE_DIR / f'tuned_model_{scope_name}.pkl')
            except Exception as e:
                print(f'\n  {name} pkl 오류: {e}')

        elapsed = round(time.time() - scope_start, 1)
        print(f'완료 ({elapsed}s)')

    # ── 캐시 저장 ──────────────────────────────────────────────────────────────
    save_json('step05_oof_arrays', oof_arrays)
    save_json('step05_oof', oof_summary)
    mark_done('step05_oof', {'scopes': list(oof_arrays.keys())})

    total_elapsed = round(time.time() - total_start, 1)
    print(f'\n✅ OOF 캐시 저장 완료 (총 {total_elapsed}s)')


[overall]  23,081행  80피처  OOF 시작...
    LR           fold 1/5  val=0.8550  (3s 경과)
    LR           fold 2/5  val=0.8533  (7s 경과)
    LR           fold 3/5  val=0.8516  (9s 경과)
    LR           fold 4/5  val=0.8526  (12s 경과)
    LR           fold 5/5  val=0.8480  (15s 경과)
  ✅ LR           OOF AUC=0.8521  gap=0.0032  과적합=False
    CatBoost     fold 1/5  val=0.8831  (81s 경과)
    CatBoost     fold 2/5  val=0.8843  (159s 경과)
    XGBoost      fold 1/5  val=0.8835  (163s 경과)
    CatBoost     fold 3/5  val=0.8802  (233s 경과)
    SVM          fold 1/5  val=0.8563  (303s 경과)
    CatBoost     fold 4/5  val=0.8818  (308s 경과)
    XGBoost      fold 2/5  val=0.8832  (314s 경과)
    CatBoost     fold 5/5  val=0.8780  (379s 경과)
  ✅ CatBoost     OOF AUC=0.8814  gap=0.0702  과적합=⚠️ True
    XGBoost      fold 3/5  val=0.8783  (455s 경과)
    RF           fold 1/5  val=0.8678  (488s 경과)
    XGBoost      fold 4/5  val=0.8811  (644s 경과)
    SVM          fold 2/5  val=0.8631  (654s 경과)
    XGBoost      fold 5/5  

In [5]:
# Cell 5: OOF 결과 표
rows = []
for scope_name in SCOPES:
    for name in MODEL_NAMES:
        info = oof_summary['by_scope'].get(scope_name, {}).get(name, {})
        rows.append({
            'scope':   scope_name,
            'model':   name,
            'OOF AUC': info.get('oof_auc'),
            'gap':     info.get('gap'),
            '과적합':  '⚠️' if info.get('overfit') else '✅',
        })

df_oof = pd.DataFrame(rows)
print('=== OOF 결과 ===')
display(df_oof.style
    .background_gradient(subset=['OOF AUC'], cmap='YlGn')
    .background_gradient(subset=['gap'], cmap='YlOrRd')
    .set_properties(**{'text-align': 'center'})
)

=== OOF 결과 ===


,scope,model,OOF AUC,gap,과적합
0,overall,LR,0.852100,0.003200,✅
1,overall,XGBoost,0.880700,0.040100,✅
2,overall,CatBoost,0.881400,0.070200,⚠️
3,overall,RF,0.864700,0.092600,⚠️
4,overall,SVM,0.855500,0.052500,⚠️
5,promotion_only,LR,0.837700,0.005600,✅
6,promotion_only,XGBoost,0.865300,0.043100,✅
7,promotion_only,CatBoost,0.864800,0.105000,⚠️
8,promotion_only,RF,0.850000,0.105800,⚠️
9,promotion_only,SVM,0.838700,0.063000,⚠️


In [6]:
# Cell 5-2: Meta LR 학습 + 캐시 저장
_meta_cached = (
    not FORCE_RERUN
    and (CACHE_DIR / 'step05_meta.json').exists()
)

if _meta_cached:
    print('캐시에서 Meta LR 로드 (재실행하려면 FORCE_RERUN=True)')
    meta_results = load_json('step05_meta')
else:
    meta_results = {'by_scope': {}}

    for scope_name, (scope_fn, _) in SCOPES.items():
        df_scope = scope_fn(train_df).reset_index(drop=True)
        y = (1 - df_scope['is_repurchase'].astype(int)).to_numpy()

        # OOF 배열로 Meta 입력 구성
        scope_arrs = oof_arrays.get(scope_name, {})
        col_names  = [n for n in MODEL_NAMES if scope_arrs.get(n) is not None]
        X_meta     = np.column_stack([np.array(scope_arrs[n]) for n in col_names])

        # best_base_auc
        base_aucs     = [oof_summary['by_scope'][scope_name][n]['oof_auc']
                         for n in col_names
                         if oof_summary['by_scope'].get(scope_name, {}).get(n, {}).get('oof_auc') is not None]
        best_base_auc = max(base_aucs) if base_aucs else 0.0

        # 5-fold CV로 Meta AUC 측정
        skf       = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        meta_oof  = np.full(len(X_meta), np.nan)
        tr_aucs, va_aucs = [], []

        for tr, va in skf.split(X_meta, y):
            clf = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
            clf.fit(X_meta[tr], y[tr])
            p_va = clf.predict_proba(X_meta[va])[:, 1]
            meta_oof[va] = p_va
            va_aucs.append(roc_auc_score(y[va], p_va))
            tr_aucs.append(roc_auc_score(y[tr], clf.predict_proba(X_meta[tr])[:, 1]))

        valid    = ~np.isnan(meta_oof)
        meta_auc = round(float(roc_auc_score(y[valid], meta_oof[valid])), 4)
        meta_gap = round(float(np.mean(tr_aucs) - np.mean(va_aucs)), 4)

        adopted = meta_auc >= best_base_auc and meta_gap <= META_GAP_MAX

        # 전체 데이터로 Meta LR 학습 + pkl 저장
        meta_full = LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
        meta_full.fit(X_meta, y)
        joblib.dump(meta_full, CACHE_DIR / f'meta_model_{scope_name}.pkl')
        save_json(f'meta_col_names_{scope_name}', col_names)

        meta_results['by_scope'][scope_name] = {
            'meta_auc':      meta_auc,
            'meta_gap':      meta_gap,
            'best_base_auc': round(best_base_auc, 4),
            'col_names':     col_names,
            'adopted':       adopted,
            'fallback':      'CatBoost' if not adopted else None,
            'note': 'Meta LR 채택' if adopted
                    else f'CatBoost fallback (meta={meta_auc} ≤ base={round(best_base_auc,4)} 또는 gap={meta_gap}>{META_GAP_MAX})',
        }

        status = '✅ 채택' if adopted else '❌ fallback'
        print(f'[{scope_name}]  meta_auc={meta_auc}  best_base={round(best_base_auc,4)}  gap={meta_gap}  {status}')

    save_json('step05_meta', meta_results)
    mark_done('step05_meta', {'scopes': list(meta_results['by_scope'].keys())})
    print('\n✅ Meta LR 캐시 저장 완료')

[overall]  meta_auc=0.8821  best_base=0.8814  gap=0.0001  ✅ 채택
[promotion_only]  meta_auc=0.8659  best_base=0.8653  gap=0.0003  ✅ 채택
[nonpromotion_only]  meta_auc=0.888  best_base=0.8872  gap=0.0001  ✅ 채택

✅ Meta LR 캐시 저장 완료


In [7]:
# Cell 5-3: OOF 최종 스코어 → step05_scores.csv 저장
score_rows = []

for scope_name, (scope_fn, _) in SCOPES.items():
    df_scope   = scope_fn(train_df).reset_index(drop=True)
    y          = (1 - df_scope['is_repurchase'].astype(int)).to_numpy()
    scope_meta = meta_results['by_scope'].get(scope_name, {})
    adopted    = scope_meta.get('adopted', False)
    col_names  = scope_meta.get('col_names', MODEL_NAMES)
    scope_arrs = oof_arrays.get(scope_name, {})

    if adopted:
        oof_cols = [np.array(scope_arrs[n]) for n in col_names if scope_arrs.get(n) is not None]
        meta_clf = joblib.load(CACHE_DIR / f'meta_model_{scope_name}.pkl')
        final_oof  = meta_clf.predict_proba(np.column_stack(oof_cols))[:, 1]
        model_used = 'meta_stacking'
    else:
        final_oof  = np.array(scope_arrs.get('CatBoost', np.zeros(len(df_scope))))
        model_used = 'catboost_fallback'

    valid   = ~np.isnan(final_oof)
    oof_auc = round(float(roc_auc_score(y[valid], final_oof[valid])), 4)

    tmp = df_scope[['USER_KEY', 'is_repurchase']].copy()
    tmp['scope']            = scope_name
    tmp['churn_risk']       = final_oof
    tmp['repurchase_score'] = 1 - final_oof
    tmp['model_used']       = model_used
    score_rows.append(tmp)

    print(f'[{scope_name}]  OOF AUC={oof_auc}  model={model_used}')

scores_df = pd.concat(score_rows, ignore_index=True)
scores_df.to_csv(CACHE_DIR / 'step05_scores.csv', index=False, encoding='utf-8-sig')

save_json('step05_score', {
    'status':   'PASS',
    'by_scope': [{'scope': s, 'rows': int((scores_df['scope'] == s).sum())} for s in SCOPES]
})
mark_done('step05_score', {'total_rows': len(scores_df)})
print(f'\n✅ step05_scores.csv 저장 완료 ({len(scores_df):,}행)')

[overall]  OOF AUC=0.8822  model=meta_stacking
[promotion_only]  OOF AUC=0.8666  model=meta_stacking
[nonpromotion_only]  OOF AUC=0.8885  model=meta_stacking

✅ step05_scores.csv 저장 완료 (46,162행)


In [8]:
# Cell 8: Test 평가 (base 5개 + 스태킹) — ⚠️ 1회만 실행할 것
# test set 재사용 시 data leakage 발생 → 이미 평가 완료 시 FORCE_RERUN=False 유지
_test_cached = (
    not FORCE_RERUN
    and (CACHE_DIR / 'test_scores.csv').exists()
)

if _test_cached:
    print('캐시에서 test 결과 로드 (재실행하려면 FORCE_RERUN=True)')
    test_results_df = pd.read_csv(CACHE_DIR / 'test_scores.csv')
    print(test_results_df.to_string(index=False))
else:
    test_result_rows = []

    for scope_name, (scope_fn, inc_promo) in SCOPES.items():
        df_test_scope  = scope_fn(test_df).reset_index(drop=True)
        df_train_scope = scope_fn(train_df)
        if len(df_test_scope) == 0:
            continue

        features = get_features(df_train_scope, inc_promo)
        X_test   = df_test_scope[features].apply(pd.to_numeric, errors='coerce').fillna(0)
        y_test   = (1 - df_test_scope['is_repurchase'].astype(int)).to_numpy()

        scope_meta = meta_results['by_scope'].get(scope_name, {})
        adopted    = scope_meta.get('adopted', False)
        col_names  = scope_meta.get('col_names', MODEL_NAMES)

        print(f'\n[{scope_name}]  {len(df_test_scope):,}행')

        # ── Base 5개 모델 test 예측 ────────────────────────────────────────────
        base_preds = {}
        for name in MODEL_NAMES:
            pkl_path = CACHE_DIR / f'stacking_{name}_{scope_name}.pkl'
            if not pkl_path.exists():
                print(f'  {name:<12} pkl 없음, 스킵')
                continue
            try:
                model = joblib.load(pkl_path)
                # sklearn 버전 불일치 방어
                try:
                    pred = model.predict_proba(X_test)[:, 1]
                except Exception:
                    from sklearn.base import clone
                    model = clone(model)
                    X_train_s = df_train_scope[features].apply(pd.to_numeric, errors='coerce').fillna(0)
                    y_train_s = (1 - df_train_scope['is_repurchase'].astype(int)).to_numpy()
                    model.fit(X_train_s, y_train_s)
                    pred = model.predict_proba(X_test)[:, 1]

                base_preds[name] = pred
                auc = round(float(roc_auc_score(y_test, pred)), 4) if len(np.unique(y_test)) == 2 else None
                f1  = round(float(f1_score(y_test, (pred >= 0.5).astype(int))), 4) if auc is not None else None
                test_result_rows.append({
                    'scope': scope_name, 'model': name, '종류': 'Base',
                    'test_auc': auc, 'test_f1': f1, '최종사용': ''
                })
                print(f'  {name:<12} AUC={auc}  F1={f1}')
            except Exception as e:
                print(f'  {name:<12} 오류: {e}')

        # ── 스태킹 or CatBoost fallback test 예측 ─────────────────────────────
        if adopted and all(n in base_preds for n in col_names):
            meta_clf      = joblib.load(CACHE_DIR / f'meta_model_{scope_name}.pkl')
            stacking_pred = meta_clf.predict_proba(
                np.column_stack([base_preds[n] for n in col_names])
            )[:, 1]
            final_label = 'meta_stacking'
        else:
            stacking_pred = base_preds.get('CatBoost')
            final_label   = 'catboost_fallback'

        if stacking_pred is not None:
            auc = round(float(roc_auc_score(y_test, stacking_pred)), 4) if len(np.unique(y_test)) == 2 else None
            f1  = round(float(f1_score(y_test, (stacking_pred >= 0.5).astype(int))), 4) if auc is not None else None
            test_result_rows.append({
                'scope': scope_name, 'model': 'Stacking', '종류': 'Meta',
                'test_auc': auc, 'test_f1': f1, '최종사용': '✅'
            })
            print(f'  {"Stacking":<12} AUC={auc}  F1={f1}  ← {final_label}')

    test_results_df = pd.DataFrame(test_result_rows)

    # ── OOF vs test 비교 표 ────────────────────────────────────────────────────
    print('\n=== OOF AUC vs Test AUC 비교 (scope=overall) ===')
    overall_oof  = oof_summary['by_scope'].get('overall', {})
    overall_meta = meta_results['by_scope'].get('overall', {})
    rows_cmp = []
    for name in MODEL_NAMES + ['Stacking']:
        if name == 'Stacking':
            oof_auc = overall_meta.get('meta_auc', '-')
        else:
            oof_auc = overall_oof.get(name, {}).get('oof_auc', '-')
        test_row = test_results_df[
            (test_results_df['scope'] == 'overall') & (test_results_df['model'] == name)
        ]
        test_auc = test_row['test_auc'].values[0] if len(test_row) > 0 else '-'
        delta = (
            round(float(test_auc) - float(oof_auc), 4)
            if oof_auc != '-' and test_auc != '-' and test_auc is not None and oof_auc is not None
            else '-'
        )
        rows_cmp.append({'모델': name, 'OOF AUC': oof_auc, 'Test AUC': test_auc, 'Δ(test-OOF)': delta})
    display(pd.DataFrame(rows_cmp))

    # ── CSV + JSON 저장 ────────────────────────────────────────────────────────
    test_results_df.to_csv(CACHE_DIR / 'test_scores.csv', index=False, encoding='utf-8-sig')

    test_summary = {
        'status': 'PASS',
        'note':   'test set 1회 평가 완료. 재실행 시 leakage 주의.',
        'by_scope': {}
    }
    for scope_name in SCOPES:
        stacking_row = test_results_df[
            (test_results_df['scope'] == scope_name) & (test_results_df['model'] == 'Stacking')
        ]
        scope_meta = meta_results['by_scope'].get(scope_name, {})
        test_summary['by_scope'][scope_name] = {
            'final_model': 'meta_stacking' if scope_meta.get('adopted') else 'catboost_fallback',
            'oof_auc':     scope_meta.get('meta_auc') if scope_meta.get('adopted') else scope_meta.get('best_base_auc'),
            'test_auc':    float(stacking_row['test_auc'].values[0]) if len(stacking_row) > 0 and stacking_row['test_auc'].values[0] is not None else None,
        }
    save_json('test_score', test_summary)

    print(f'\n✅ test_scores.csv 저장 완료 ({len(test_results_df)}행)')
    print('⚠️  test set은 1회 평가 완료. 결과 보고 재튜닝 시 data leakage.')


[overall]  2,545행
  LR           AUC=0.8508  F1=0.6437
  XGBoost      AUC=0.8855  F1=0.7036
  CatBoost     AUC=0.8861  F1=0.7068
  RF           AUC=0.8678  F1=0.676
  SVM          AUC=0.8595  F1=0.6322
  Stacking     AUC=0.886  F1=0.6696  ← meta_stacking

[promotion_only]  1,293행
  LR           AUC=0.8467  F1=0.67
  XGBoost      AUC=0.8814  F1=0.7257
  CatBoost     AUC=0.8836  F1=0.7309
  RF           AUC=0.866  F1=0.7104
  SVM          AUC=0.8536  F1=0.6659
  Stacking     AUC=0.8824  F1=0.6989  ← meta_stacking

[nonpromotion_only]  1,252행
  LR           AUC=0.8432  F1=0.584
  XGBoost      AUC=0.8751  F1=0.646
  CatBoost     AUC=0.8737  F1=0.63
  RF           AUC=0.8565  F1=0.5953
  SVM          AUC=0.8487  F1=0.56
  Stacking     AUC=0.8759  F1=0.6159  ← meta_stacking

=== OOF AUC vs Test AUC 비교 (scope=overall) ===


,모델,OOF AUC,Test AUC,Δ(test-OOF)
0,LR,0.8521,0.8508,-0.0013
1,XGBoost,0.8807,0.8855,0.0048
2,CatBoost,0.8814,0.8861,0.0047
3,RF,0.8647,0.8678,0.0031
4,SVM,0.8555,0.8595,0.0040
5,Stacking,0.8821,0.8860,0.0039



✅ test_scores.csv 저장 완료 (18행)
⚠️  test set은 1회 평가 완료. 결과 보고 재튜닝 시 data leakage.


In [9]:
# Cell 9: 최종 요약
print('=' * 90)
print('최종 성능 요약')
print('=' * 90)

summary_rows = []

for scope_name in SCOPES:
    scope_meta  = meta_results['by_scope'].get(scope_name, {})
    adopted     = scope_meta.get('adopted', False)
    final_model = 'Meta Stacking' if adopted else 'CatBoost fallback'

    print(f'\n[{scope_name}]  최종 모델: {final_model}')
    print(f'  {"모델":<14} {"종류":>5} {"OOF AUC":>9} {"gap":>7} {"과적합":>6} {"test AUC":>10} {"최종"}')
    print(f'  {"-" * 60}')

    for name in MODEL_NAMES:
        oof_info = oof_summary['by_scope'].get(scope_name, {}).get(name, {})
        test_row = test_results_df[
            (test_results_df['scope'] == scope_name) & (test_results_df['model'] == name)
        ]
        test_auc = test_row['test_auc'].values[0] if len(test_row) > 0 else '-'
        oof_auc  = oof_info.get('oof_auc', '-')
        gap      = oof_info.get('gap', '-')
        overfit  = '⚠️' if oof_info.get('overfit') else '✅'
        print(f'  {name:<14} {"Base":>5} {str(oof_auc):>9} {str(gap):>7} {overfit:>6} {str(test_auc):>10}')

        summary_rows.append({
            'scope': scope_name, 'model': name, '종류': 'Base',
            'OOF AUC': oof_auc, 'gap': gap,
            '과적합': '⚠️' if oof_info.get('overfit') else '✅',
            'test AUC': test_auc, '최종사용': ''
        })

    # 스태킹 행
    stack_row = test_results_df[
        (test_results_df['scope'] == scope_name) & (test_results_df['model'] == 'Stacking')
    ]
    if len(stack_row) > 0:
        meta_auc = scope_meta.get('meta_auc', '-')
        meta_gap = scope_meta.get('meta_gap', '-')
        test_auc = stack_row['test_auc'].values[0]
        flag     = '✅ 채택' if adopted else '❌ fallback'
        print(f'  {"Stacking":<14} {"Meta":>5} {str(meta_auc):>9} {str(meta_gap):>7} {"":>6} {str(test_auc):>10}  ← {flag}')

        summary_rows.append({
            'scope': scope_name, 'model': 'Stacking', '종류': 'Meta',
            'OOF AUC': meta_auc, 'gap': meta_gap, '과적합': '',
            'test AUC': test_auc, '최종사용': flag
        })

# 표 형태로도 출력
print('\n' + '=' * 90)
print('DataFrame 요약')
display(pd.DataFrame(summary_rows).style
    .background_gradient(subset=['OOF AUC', 'test AUC'], cmap='YlGn')
    .set_properties(**{'text-align': 'center'})
)

# 캐시 저장 현황
print('\n' + '=' * 90)
print('캐시 저장 현황')
sf = CACHE_DIR / 'pipeline_state.json'
if sf.exists():
    state = json.loads(sf.read_text(encoding='utf-8'))
    for step, info in state.items():
        if 'step05' in step:
            print(f'  ✅ {step:<25} {info["completed_at"][:19]}')

pkl_files = (list(CACHE_DIR.glob('stacking_*.pkl'))
             + list(CACHE_DIR.glob('meta_model_*.pkl'))
             + list(CACHE_DIR.glob('tuned_model_*.pkl')))
print(f'\npkl 파일: {len(pkl_files)}개')
print(f'step05_scores.csv: {(CACHE_DIR / "step05_scores.csv").exists()}')
print(f'test_scores.csv:   {(CACHE_DIR / "test_scores.csv").exists()}')

최종 성능 요약

[overall]  최종 모델: Meta Stacking
  모델                종류   OOF AUC     gap    과적합   test AUC 최종
  ------------------------------------------------------------
  LR              Base    0.8521  0.0032      ✅     0.8508
  XGBoost         Base    0.8807  0.0401      ✅     0.8855
  CatBoost        Base    0.8814  0.0702     ⚠️     0.8861
  RF              Base    0.8647  0.0926     ⚠️     0.8678
  SVM             Base    0.8555  0.0525     ⚠️     0.8595
  Stacking        Meta    0.8821  0.0001             0.886  ← ✅ 채택

[promotion_only]  최종 모델: Meta Stacking
  모델                종류   OOF AUC     gap    과적합   test AUC 최종
  ------------------------------------------------------------
  LR              Base    0.8377  0.0056      ✅     0.8467
  XGBoost         Base    0.8653  0.0431      ✅     0.8814
  CatBoost        Base    0.8648   0.105     ⚠️     0.8836
  RF              Base      0.85  0.1058     ⚠️      0.866
  SVM             Base    0.8387   0.063     ⚠️     0.8536
  Stacking 

,scope,model,종류,OOF AUC,gap,과적합,test AUC,최종사용
0,overall,LR,Base,0.852100,0.003200,✅,0.850800,
1,overall,XGBoost,Base,0.880700,0.040100,✅,0.885500,
2,overall,CatBoost,Base,0.881400,0.070200,⚠️,0.886100,
3,overall,RF,Base,0.864700,0.092600,⚠️,0.867800,
4,overall,SVM,Base,0.855500,0.052500,⚠️,0.859500,
5,overall,Stacking,Meta,0.882100,0.000100,,0.886000,✅ 채택
6,promotion_only,LR,Base,0.837700,0.005600,✅,0.846700,
7,promotion_only,XGBoost,Base,0.865300,0.043100,✅,0.881400,
8,promotion_only,CatBoost,Base,0.864800,0.105000,⚠️,0.883600,
9,promotion_only,RF,Base,0.850000,0.105800,⚠️,0.866000,



캐시 저장 현황
  ✅ step05_oof                2026-06-05T16:45:51
  ✅ step05_meta               2026-06-05T16:45:54
  ✅ step05_score              2026-06-05T16:45:55

pkl 파일: 21개
step05_scores.csv: True
test_scores.csv:   True
